# AI-Based Resume Screening and Candidate Shortlisting System
**Google Colab notebook: data, baseline ML, semantic matching, evaluation.**

Before you start: `Runtime -> Change runtime type -> T4 GPU` (makes embeddings much faster), then run the cells top to bottom.

## Step 1: Install libraries
Colab already has pandas, scikit-learn, matplotlib and torch. We add Sentence-Transformers (semantic embeddings), pdfplumber (PDF reading) and python-docx (Word reading).

In [ ]:
!pip -q install sentence-transformers pdfplumber python-docx

## Step 2: Connect to Kaggle
1. Log in to [kaggle.com](https://www.kaggle.com) -> profile picture -> **Settings** -> **API** -> **Create New Token**. A file named `kaggle.json` downloads.
2. Run the cell below and choose that file when asked.

If your Kaggle account shows a token *string* instead of a file, skip the upload and run
`import os; os.environ["KAGGLE_API_TOKEN"] = "<your token>"` (supported by newer Kaggle CLI versions).

Never share `kaggle.json` and never commit it to GitHub.

In [ ]:
from google.colab import files
uploaded = files.upload()          # select kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

## Step 3: Download the dataset
We use the **Resume Dataset** (Livecareer): about 2,400 resumes in 24 job categories (`snehaanbhawal/resume-dataset`).

*Fallback if the API fails:* download the zip manually from the dataset page on Kaggle, upload it with `files.upload()`, then run `!unzip -q resume-dataset.zip -d data`.

In [ ]:
!kaggle datasets download -d snehaanbhawal/resume-dataset -p data --unzip
!ls data

## Step 4: Load and explore the data

In [ ]:
import glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

csv_path = glob.glob("data/**/Resume.csv", recursive=True)[0]
df = pd.read_csv(csv_path)
df = df.rename(columns={"Resume_str": "resume"})[["resume", "Category"]].dropna()
print("Shape:", df.shape)
df.head()

In [ ]:
df["Category"].value_counts().sort_values().plot(kind="barh", figsize=(7, 7), title="Resumes per category")
plt.show()

df["n_words"] = df["resume"].str.split().str.len()
df["n_words"].describe()

## Step 5: Clean the text
Remove any HTML tags and collapse whitespace. We keep the original casing for the language model and create a lowercase copy for TF-IDF.

In [ ]:
def clean_text(t):
    t = re.sub(r"<[^>]+>", " ", str(t))
    return re.sub(r"\s+", " ", t).strip()

df["resume"] = df["resume"].apply(clean_text)
df = df[df["resume"].str.len() > 100].reset_index(drop=True)
df["clean_lower"] = df["resume"].str.lower()
print(len(df), "resumes after cleaning")

## Step 6: Baseline ML model: resume category classifier
A classic **TF-IDF + Logistic Regression** model predicts a resume's job category. It is your first trained ML model and a useful baseline for the report.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["Category"], random_state=42)

tfidf_clf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, max_features=30000)
X_train = tfidf_clf.fit_transform(train_df["clean_lower"])
X_test = tfidf_clf.transform(test_df["clean_lower"])

clf = LogisticRegression(max_iter=1000, C=10).fit(X_train, train_df["Category"])
pred = clf.predict(X_test)
print("Accuracy:", round(accuracy_score(test_df["Category"], pred), 3))
print(classification_report(test_df["Category"], pred, zero_division=0))

## Step 7: Build the screening engine (`screening.py`)
This file is the heart of the project. It contains:

| Function | Purpose |
|---|---|
| `extract_skills` | finds skills using a taxonomy with aliases ("ML" -> machine learning) |
| `extract_years`, `education_level` | estimates experience and highest degree |
| `mask_pii` | blind screening: hides emails, phones, links, pronouns |
| `embed_resume_chunks`, `jd_embedding`, `semantic_raw` | Sentence-BERT semantic similarity |
| `score_candidates` | hybrid weighted score, ranking, matched/missing skills |
| `read_file` | reads PDF, DOCX, TXT |

The cell writes the file to disk so the web app can reuse the **same** code you evaluate here.

In [ ]:
%%writefile screening.py
"""screening.py - core engine for the AI Resume Screening & Shortlisting system.

Pipeline: read file -> (optional) mask PII -> extract skills / years / education
-> semantic similarity (Sentence-BERT) -> hybrid weighted score -> ranked table.
"""
import io
import re
from datetime import datetime

import numpy as np
import pandas as pd

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
_model = None


def get_model():
    """Load the Sentence-BERT model once (lazy import keeps helpers lightweight)."""
    global _model
    if _model is None:
        from sentence_transformers import SentenceTransformer
        _model = SentenceTransformer(MODEL_NAME)
    return _model


# ----------------------------------------------------------------------------
# 1. Skill taxonomy: canonical name -> aliases. Extend this freely for your domain.
# ----------------------------------------------------------------------------
SKILLS = {
    # programming
    "python": [], "java": [], "javascript": ["js"], "typescript": [],
    "c++": ["cpp"], "c#": [], "php": [], "ruby": [], "kotlin": [], "swift": [],
    "r programming": ["r language"], "matlab": [], "scala": [], "html": ["html5"],
    "css": ["css3"], "bash": ["shell scripting"],
    # web / frameworks
    "react": ["reactjs", "react.js"], "angular": ["angularjs"], "vue": ["vue.js", "vuejs"],
    "node.js": ["nodejs", "node js"], "django": [], "flask": [], "fastapi": [],
    "spring boot": ["springboot"], ".net": ["dotnet", "asp.net"],
    # data & databases
    "sql": ["t-sql", "pl/sql"], "mysql": [], "postgresql": ["postgres"],
    "mongodb": ["mongo"], "oracle": [], "nosql": [], "excel": ["ms excel", "microsoft excel"],
    "power bi": ["powerbi"], "tableau": [], "data analysis": ["data analytics"],
    "data visualization": ["data visualisation"], "etl": [], "hadoop": [],
    "spark": ["pyspark", "apache spark"], "kafka": [],
    # machine learning
    "machine learning": ["ml"], "deep learning": [],
    "nlp": ["natural language processing"], "computer vision": [],
    "tensorflow": [], "pytorch": [], "keras": [], "scikit-learn": ["sklearn", "scikit learn"],
    "pandas": [], "numpy": [], "statistics": ["statistical analysis"],
    "llm": ["llms", "large language models"],
    # cloud / devops
    "aws": ["amazon web services"], "azure": [], "gcp": ["google cloud"],
    "docker": [], "kubernetes": ["k8s"], "git": ["github", "gitlab"],
    "ci/cd": ["jenkins"], "linux": [], "rest api": ["restful", "rest apis", "restful api"],
    "microservices": [], "agile": ["scrum"], "jira": [],
    "networking": ["network administration"], "technical support": ["help desk", "helpdesk"],
    # business / management
    "project management": [], "leadership": [], "communication": [],
    "customer service": [], "sales": [], "marketing": [], "digital marketing": [],
    "seo": [], "negotiation": [], "business development": [],
    "budgeting": ["budget management"], "financial reporting": [],
    "accounting": [], "accounts payable": [], "accounts receivable": [],
    "general ledger": [], "auditing": ["audit"], "taxation": ["tax preparation"],
    "payroll": [], "quickbooks": [], "sap": [], "tally": [],
    # HR
    "recruitment": ["recruiting", "talent acquisition"], "onboarding": [],
    "employee relations": [], "performance management": [], "hris": [],
    "benefits administration": [], "training": ["training and development"],
    # engineering / operations
    "autocad": [], "solidworks": [], "cad": [], "plc": [], "six sigma": [],
    "quality control": ["quality assurance"], "project planning": [],
    "manufacturing": [], "supply chain": [], "inventory management": [],
    # kitchen / hospitality / healthcare
    "menu planning": [], "food safety": ["haccp"], "catering": [],
    "kitchen management": [], "patient care": [],
}
ALL_SKILLS = sorted(SKILLS)


def _build_patterns():
    pats = {}
    for canon, aliases in SKILLS.items():
        names = sorted({canon, *aliases}, key=len, reverse=True)
        body = "|".join(re.escape(n) for n in names)
        pats[canon] = re.compile(r"(?<![a-z0-9+#.])(?:" + body + r")(?![a-z0-9+#])")
    return pats


_PATTERNS = _build_patterns()


def extract_skills(text):
    """Return the set of canonical skills mentioned in `text`."""
    t = str(text).lower()
    return {canon for canon, pat in _PATTERNS.items() if pat.search(t)}


# ----------------------------------------------------------------------------
# 2. Experience, education, extras
# ----------------------------------------------------------------------------
def extract_years(text):
    """Estimate years of experience: explicit '5 years' claims, else date ranges."""
    t = str(text).lower()
    explicit = [int(x) for x in re.findall(r"(\d{1,2})\+?\s*(?:years|yrs)", t)]
    explicit = [x for x in explicit if 0 < x <= 40]
    if explicit:
        return float(max(explicit))
    now, total = datetime.now().year, 0.0
    pattern = (r"((?:19|20)\d{2})\s*(?:-|to|\u2013|\u2014)\s*(?:\d{1,2}/)?"
               r"((?:19|20)\d{2}|present|current|now)")
    for start, end in re.findall(pattern, t):
        s = int(start)
        e = now if end in ("present", "current", "now") else int(end)
        if 0 <= e - s <= 40:
            total += e - s
    return float(min(total, 40))


EDU_REQ = {"Any": 0, "Diploma": 1, "Bachelor": 2, "Master": 3, "PhD": 4}


def education_level(text):
    """0 = none found, 1 = diploma, 2 = bachelor, 3 = master, 4 = PhD."""
    t = str(text).lower()
    if re.search(r"\b(ph\.?d|doctorate|doctoral)\b", t):
        return 4
    if re.search(r"\b(master'?s?|mba|m\.?tech|msc|m\.sc)\b", t):
        return 3
    if re.search(r"\b(bachelor'?s?|b\.?tech|bsc|b\.sc|bba|bca|b\.e\.)", t):
        return 2
    if re.search(r"\b(diploma|associate'?s?)\b", t):
        return 1
    return 0


_EXTRA_RE = re.compile(r"certif|award|publication|patent|project|hackathon|open[- ]source")

# ----------------------------------------------------------------------------
# 3. Bias mitigation: blind screening
# ----------------------------------------------------------------------------
def mask_pii(text):
    """Mask contact details and gendered words. (Names need NER - see README.)"""
    t = str(text)
    t = re.sub(r"[\w.+-]+@[\w-]+\.[\w.-]+", "[EMAIL]", t)
    t = re.sub(r"(?:https?://|www\.)\S+|linkedin\.com/\S+|github\.com/\S+", "[URL]", t, flags=re.I)
    t = re.sub(r"(?<!\d)(?:\+?\d{1,3}[\s.-]?)?\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}(?!\d)", "[PHONE]", t)
    t = re.sub(r"\b(he|she|his|hers?|him|mr|mrs|ms|miss)\b\.?", "[X]", t, flags=re.I)
    return t


# ----------------------------------------------------------------------------
# 4. Semantic similarity with Sentence-BERT (chunked so long resumes are covered)
# ----------------------------------------------------------------------------
def chunk_text(text, size=120, overlap=20):
    words = str(text).split()
    if not words:
        return [""]
    return [" ".join(words[i:i + size])
            for i in range(0, max(len(words) - overlap, 1), size - overlap)]


def embed_resume_chunks(texts, batch_size=64, show_progress=False):
    """Return one (n_chunks, dim) array per resume. Reuse it across many JDs."""
    model = get_model()
    all_chunks, counts = [], []
    for t in texts:
        c = chunk_text(t)
        all_chunks.extend(c)
        counts.append(len(c))
    embs = model.encode(all_chunks, batch_size=batch_size, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=show_progress)
    out, pos = [], 0
    for n in counts:
        out.append(embs[pos:pos + n])
        pos += n
    return out


def jd_embedding(jd):
    e = get_model().encode(chunk_text(jd), normalize_embeddings=True,
                           convert_to_numpy=True, show_progress_bar=False)
    v = e.mean(axis=0)
    return v / (np.linalg.norm(v) + 1e-9)


def semantic_raw(jd_vec, chunk_embs, top_k=3):
    """Per resume: mean of the top-k chunk similarities to the JD."""
    return np.array([float(np.sort(e @ jd_vec)[-top_k:].mean()) for e in chunk_embs])


SIM_LOW, SIM_HIGH = 0.10, 0.60   # raw cosine range mapped to 0..1


# ----------------------------------------------------------------------------
# 5. Hybrid scoring + ranking
# ----------------------------------------------------------------------------
DEFAULT_WEIGHTS = {"semantic": 0.30, "skills": 0.35, "experience": 0.20,
                   "education": 0.10, "extras": 0.05}


def label_for(score):
    return "Strong Match" if score >= 75 else "Moderate Match" if score >= 50 else "Weak Match"


def score_candidates(resumes, jd, names=None, must=None, nice=None, min_years=0,
                     min_education="Any", weights=None, blind=False, chunk_embs=None):
    """Score every resume against the JD and return a ranked DataFrame."""
    n = len(resumes)
    if blind:
        names = [f"Candidate {i + 1}" for i in range(n)]
    elif names is None:
        names = [f"Candidate {i + 1}" for i in range(n)]

    w = {**DEFAULT_WEIGHTS, **(weights or {})}
    total = sum(w.values()) or 1.0
    w = {k: v / total for k, v in w.items()}

    texts = [mask_pii(r) for r in resumes] if blind else list(resumes)
    if blind or chunk_embs is None:          # embeddings must match the (masked) text
        chunk_embs = embed_resume_chunks(texts)
    raw = semantic_raw(jd_embedding(jd), chunk_embs)
    semantic = np.clip((raw - SIM_LOW) / (SIM_HIGH - SIM_LOW), 0, 1)

    jd_skills = extract_skills(jd)
    must = {s.lower() for s in must} if must else set(jd_skills)
    nice = {s.lower() for s in nice} if nice else (jd_skills - must)
    denom = 2 * len(must) + len(nice)
    req_edu = EDU_REQ.get(min_education, 0)

    rows = []
    for i, t in enumerate(texts):
        sk = extract_skills(t)
        skill = (2 * len(must & sk) + len(nice & sk)) / denom if denom else float(semantic[i])
        yrs = extract_years(t)
        exp = 1.0 if min_years <= 0 else min(1.0, yrs / min_years)
        edu = 1.0 if req_edu == 0 else min(1.0, education_level(t) / req_edu)
        extras = min(1.0, len(_EXTRA_RE.findall(t.lower())) / 5)
        final = 100 * (w["semantic"] * semantic[i] + w["skills"] * skill +
                       w["experience"] * exp + w["education"] * edu + w["extras"] * extras)
        rows.append({
            "idx": i, "candidate": names[i], "final_score": round(final, 1),
            "match": label_for(final), "semantic": round(float(semantic[i]), 3),
            "skills": round(skill, 3), "experience": round(exp, 3),
            "education": round(edu, 3), "extras": round(extras, 3),
            "years_exp": yrs,
            "matched_skills": ", ".join(sorted((must | nice) & sk)),
            "missing_skills": ", ".join(sorted((must | nice) - sk)),
        })
    return (pd.DataFrame(rows).sort_values("final_score", ascending=False)
            .reset_index(drop=True))


# ----------------------------------------------------------------------------
# 6. Reading uploaded files (PDF / DOCX / TXT)
# ----------------------------------------------------------------------------
def read_file(name, data):
    """Extract text from an uploaded file's bytes."""
    ext = name.lower().rsplit(".", 1)[-1]
    if ext == "pdf":
        import pdfplumber
        with pdfplumber.open(io.BytesIO(data)) as pdf:
            return "\n".join((p.extract_text() or "") for p in pdf.pages)
    if ext == "docx":
        import docx
        d = docx.Document(io.BytesIO(data))
        parts = [p.text for p in d.paragraphs]
        for table in d.tables:
            for row in table.rows:
                parts.extend(cell.text for cell in row.cells)
        return "\n".join(parts)
    return data.decode("utf-8", errors="ignore")

### Quick unit checks
Confirm each helper works before running the heavy model.

In [ ]:
import importlib, screening
importlib.reload(screening)
from screening import *

sample_resume = df["resume"].iloc[0]
print("Skills:", extract_skills(sample_resume))
print("Years :", extract_years(sample_resume))
print("Edu   :", education_level(sample_resume))
print(mask_pii("Reach John at john@mail.com or +1 415-555-2671. She has 5 years of experience."))

## Step 8: Create embeddings for a sample of resumes
Each resume is split into ~120-word chunks (the model reads about 200 words at a time) and every chunk is embedded once. We reuse these embeddings for every job description we test.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())

sample = df.sample(n=min(800, len(df)), random_state=42).reset_index(drop=True)
chunk_embs = embed_resume_chunks(sample["resume"].tolist(), show_progress=True)
print("Embedded", len(chunk_embs), "resumes")

## Step 9: Evaluate: TF-IDF vs SBERT vs Hybrid
We write one job description per category and treat resumes from the same category as "relevant". This is a **proxy** for recruiter judgement (for your final report, also label a small gold set by hand).

Metrics: **Precision@10** (how many of the top 10 are relevant) and **NDCG@10** (are relevant ones ranked higher?).

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

JDS = {
    "INFORMATION-TECHNOLOGY": "Software engineer with strong Python, Java, SQL and AWS cloud experience. Build and maintain web applications, REST APIs and databases. Linux, Git, Docker, networking and technical support experience is a plus.",
    "HR": "Human resources specialist to manage recruitment, onboarding, employee relations, payroll and performance management. Experience with HRIS and benefits administration.",
    "ACCOUNTANT": "Accountant responsible for accounts payable, accounts receivable, general ledger, financial reporting, budgeting, auditing and taxation. Experience with QuickBooks, SAP and Excel.",
    "CHEF": "Chef to lead kitchen operations: menu planning, food preparation, food safety (HACCP), inventory management and catering while supervising kitchen staff.",
    "ENGINEERING": "Engineer with experience in design, AutoCAD, SolidWorks, quality control, project planning and manufacturing. Six Sigma and PLC experience preferred.",
}

def precision_at_k(rel, k=10):
    return float(np.mean(rel[:k]))

def ndcg_at_k(rel, k=10):
    rel = np.asarray(rel, dtype=float)
    dcg = sum(r / np.log2(i + 2) for i, r in enumerate(rel[:k]))
    ideal = sorted(rel, reverse=True)
    idcg = sum(r / np.log2(i + 2) for i, r in enumerate(ideal[:k]))
    return float(dcg / idcg) if idcg else 0.0

tfidf_ret = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=30000)
X_sample = tfidf_ret.fit_transform(sample["resume"].str.lower())
texts = sample["resume"].tolist()

rows = []
for cat, jd_text in JDS.items():
    truth = (sample["Category"] == cat).values.astype(int)

    # 1) TF-IDF baseline
    sims = cosine_similarity(tfidf_ret.transform([jd_text.lower()]), X_sample).ravel()
    rel = truth[np.argsort(-sims)]
    rows.append(("TF-IDF", cat, precision_at_k(rel), ndcg_at_k(rel)))

    # 2) SBERT only
    res = score_candidates(texts, jd_text, chunk_embs=chunk_embs,
                           weights={"semantic": 1, "skills": 0, "experience": 0, "education": 0, "extras": 0})
    rel = truth[res["idx"].values]
    rows.append(("SBERT only", cat, precision_at_k(rel), ndcg_at_k(rel)))

    # 3) Hybrid (default weights)
    res = score_candidates(texts, jd_text, chunk_embs=chunk_embs)
    rel = truth[res["idx"].values]
    rows.append(("Hybrid", cat, precision_at_k(rel), ndcg_at_k(rel)))

results = pd.DataFrame(rows, columns=["method", "category", "P@10", "NDCG@10"])
print("Average over all job categories:")
display(results.groupby("method")[["P@10", "NDCG@10"]].mean().round(3))
print("Per category (NDCG@10):")
display(results.pivot(index="category", columns="method", values="NDCG@10").round(3))

**Copy the averages table into your report and README.** Expect SBERT and Hybrid to beat plain TF-IDF on vocabulary-mismatch cases. If Hybrid is not always better, that is a genuine finding: the skill taxonomy is biased toward tech skills, so extending it is a good "future work" item.

## Step 10: Try it on a real job description

In [ ]:
jd = """
We are hiring a Data Analyst with 3+ years of experience. Must know Python, SQL and Excel.
Experience with Tableau or Power BI, statistics and data visualization is a plus.
Bachelor's degree required.
"""

names = [f"{c}-{i}" for i, c in enumerate(sample["Category"])]
top = score_candidates(texts, jd, names=names, nice=["tableau", "power bi", "statistics", "data visualization"],
                       must=["python", "sql", "excel"], min_years=3, min_education="Bachelor",
                       chunk_embs=chunk_embs)
top[["candidate", "final_score", "match", "years_exp", "matched_skills", "missing_skills"]].head(10)

## Step 11: Bias check: blind screening
Blind mode masks contact info and pronouns. A good fair system should change scores only slightly, because scoring should depend on skills, not identity markers.

In [ ]:
subset = sample.head(60)
a = score_candidates(subset["resume"].tolist(), jd, blind=False).set_index("idx")["final_score"]
b = score_candidates(subset["resume"].tolist(), jd, blind=True).set_index("idx")["final_score"]
diff = (a - b.reindex(a.index)).abs()
print("Mean absolute score change with blind mode:", round(diff.mean(), 2), "points (max", round(diff.max(), 2), ")")

## Step 12: Test with your own resume files (PDF / DOCX / TXT)
Upload one or more resumes to see the full pipeline, including file reading.

In [ ]:
up = files.upload()
names_up, texts_up = [], []
for name, data in up.items():
    names_up.append(name)
    texts_up.append(read_file(name, data))

score_candidates(texts_up, jd, names=names_up)[
    ["candidate", "final_score", "match", "matched_skills", "missing_skills"]]

## Step 13: Save and download files for the web app
Download `screening.py` (goes into your GitHub repo next to `app.py`) and your results.

In [ ]:
results.to_csv("evaluation_results.csv", index=False)
top.to_csv("sample_shortlist.csv", index=False)

files.download("screening.py")
files.download("evaluation_results.csv")

## Done. Next steps
1. Save this notebook to GitHub: `File -> Save a copy in GitHub`.
2. Build the web frontend (`app.py`), push the project to GitHub and deploy. The companion guide covers all three.